In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D7 — Delivering STEM Skills for the Economy
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!pip install PyMuPDF

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import platform
import re
import sys

import fitz
import pandas as pd

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D7"

DOCUMENT_NAME = (
    "UK National Audit Office — "
    "Delivering STEM skills for the economy"
)

BRANCH = "A"

BRANCH_NAME = "Direct Ingestion"

SOURCE_FORMAT = ".pdf"

INPUT_REPRESENTATION = "Original PDF document"

DIRECT_DOCUMENT_INGESTION = True

EXPECTED_PAGE_COUNT = 12


EXPECTED_RECORD_COUNT = 59

EXPECTED_CATEGORY_COUNTS = {
    "Key fact": 10,
    "Policy context": 3,
    "Policy finding": 7,
    "Education pipeline statistic": 24,
    "Government initiative": 9,
    "Recommendation": 6
}


# ------------------------------------------------------------
# Fixed Stage 1 extraction schema
# ------------------------------------------------------------

EXPECTED_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]


STRING_OR_NULL_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]


NUMERIC_OR_NULL_FIELDS = [
    "Value"
]

# This is separate from JSON type validity.
MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Source Location"
]


ALLOWED_CATEGORIES = set(
    EXPECTED_CATEGORY_COUNTS
)


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "outputs_D7_branch_A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


INPUT_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D7_branch_A_input_integrity.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D7_branch_A_representation.json"
)

PROMPT_PATH = (
    OUTPUT_DIR
    / "D7_branch_A_prompt.txt"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D7_branch_A_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D7_branch_A_parsed_extraction.json"
)

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D7_branch_A_technical_diagnostics.json"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D7_branch_A_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D7_branch_A_experiment_summary.json"
)


print(
    "Document:",
    DOCUMENT_ID
)

print(
    "Branch:",
    BRANCH
)

print(
    "Input representation:",
    INPUT_REPRESENTATION
)

print(
    "Expected records:",
    EXPECTED_RECORD_COUNT
)

print(
    "Expected fields:",
    len(EXPECTED_FIELDS)
)

In [ ]:
# ============================================================
# 2. Source document and integrity diagnostics
# ============================================================

print(
    "Upload the original D7 PDF."
)

uploaded = files.upload()


pdf_paths = [
    Path(name)

    for name
    in uploaded

    if name.lower().endswith(
        ".pdf"
    )
]


if len(pdf_paths) != 1:

    raise ValueError(
        "Upload exactly one PDF file."
    )


SOURCE_PATH = pdf_paths[0]


def sha256_file(path):

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)


FILE_SIZE_BYTES = (
    SOURCE_PATH.stat().st_size
)

FILE_NON_EMPTY = (
    FILE_SIZE_BYTES > 0
)


pdf_document = fitz.open(
    SOURCE_PATH
)


PAGE_COUNT = len(
    pdf_document
)

PAGE_COUNT_VALID = (
    PAGE_COUNT
    == EXPECTED_PAGE_COUNT
)


page_rows = []

full_document_text_parts = []


for page_number, page in enumerate(
    pdf_document,
    start=1
):

    text = (
        page.get_text(
            "text"
        )
        or ""
    )

    full_document_text_parts.append(
        text
    )

    page_rows.append(
        {
            "Page Number":
                page_number,

            "Character Count":
                len(
                    text
                ),

            "Word Count":
                len(
                    text.split()
                ),

            "Text Extractable":
                bool(
                    text.strip()
                )
        }
    )


page_df = pd.DataFrame(
    page_rows
)


TEXT_EXTRACTABLE = bool(
    page_df[
        "Text Extractable"
    ].all()
)


OCR_REQUIRED = (
    not TEXT_EXTRACTABLE
)


FULL_DOCUMENT_TEXT = "\n".join(
    full_document_text_parts
)


# ------------------------------------------------------------
# Expected document-component diagnostics
# ------------------------------------------------------------

GROUNDING_MARKERS = {
    "key_facts":
        "Key facts",

    "summary":
        "Summary",

    "government_intervention":
        "Government intervention",

    "key_findings":
        "Key findings",

    "education_pipeline":
        (
            "The performance of the education "
            "pipeline in delivering STEM skills"
        ),

    "latest_initiatives":
        (
            "The latest initiatives designed "
            "to enhance the development of STEM skills"
        ),

    "value_for_money_conclusion":
        "Conclusion on value for money",

    "recommendations":
        "Recommendations"
}


GROUNDING_MARKER_STATUS = {
    key: (
        marker
        in FULL_DOCUMENT_TEXT
    )

    for key, marker
    in GROUNDING_MARKERS.items()
}


DOCUMENT_GROUNDING_VALID = all(
    GROUNDING_MARKER_STATUS.values()
)


DIRECT_PDF_INGESTION_USABLE = all([
    FILE_NON_EMPTY,
    PAGE_COUNT_VALID,
    TEXT_EXTRACTABLE,
    DOCUMENT_GROUNDING_VALID
])


INPUT_INTEGRITY_PASSED = (
    DIRECT_PDF_INGESTION_USABLE
)


INPUT_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_file":
        SOURCE_PATH.name,

    "input_file_sha256":
        SOURCE_SHA256,

    "input_representation":
        INPUT_REPRESENTATION,

    "file_size_bytes":
        FILE_SIZE_BYTES,

    "file_non_empty":
        FILE_NON_EMPTY,

    "page_count":
        PAGE_COUNT,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "page_count_valid":
        PAGE_COUNT_VALID,

    "text_layer_present":
        TEXT_EXTRACTABLE,

    "ocr_required":
        OCR_REQUIRED,

    "grounding_marker_checks":
        GROUNDING_MARKER_STATUS,

    "all_expected_components_present":
        DOCUMENT_GROUNDING_VALID,

    "direct_pdf_ingestion_usable":
        DIRECT_PDF_INGESTION_USABLE,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED
}


INPUT_INTEGRITY_PATH.write_text(
    json.dumps(
        INPUT_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "Source:",
    SOURCE_PATH.name
)

print(
    "SHA-256:",
    SOURCE_SHA256
)

print(
    "Page count:",
    PAGE_COUNT
)

display(
    page_df
)

print(
    "\nGrounding markers:"
)

print(
    json.dumps(
        GROUNDING_MARKER_STATUS,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "\nInput integrity passed:",
    INPUT_INTEGRITY_PASSED
)

if not PAGE_COUNT_VALID:

    raise AssertionError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, "
        f"found {PAGE_COUNT}."
    )


if not TEXT_EXTRACTABLE:

    raise AssertionError(
        "The source PDF does not contain "
        "extractable text."
    )


if not DOCUMENT_GROUNDING_VALID:

    raise AssertionError(
        "One or more expected D7 document "
        "components were not detected."
    )

In [ ]:
# ============================================================
# 3. Branch A representation
# ============================================================

REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Original source document",

    "input_file":
        SOURCE_PATH.name,

    "input_format":
        SOURCE_FORMAT,

    "diagnostic_pdf_text_inspection_applied":
        True,

    "pdf_to_text_conversion_applied":
        False,

    "derived_representation_used_as_model_input":
        False,

    "ocr_applied":
        False,

    "page_cropping_applied":
        False,

    "page_extraction_applied":
        False,

    "layout_reconstruction_applied":
        False,

    "table_conversion_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "complete_original_pdf_supplied":
        True,

    "model_input_description": (
        "The complete original 12-page D7 PDF is "
        "submitted directly to the LLM. PyMuPDF text "
        "extraction is used only for source-integrity "
        "diagnostics and is not supplied to the model "
        "as an alternative representation."
    )
}


REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 4. Extraction prompt
# ============================================================

BRANCH_A_PROMPT = """You are an information extraction assistant.

Extract the policy and quantitative records represented within the
defined scope of the attached original PDF report:

“Delivering STEM (science, technology, engineering and mathematics)
skills for the economy”.

Treat the attached original PDF as the only source of information.

Include records from the following defined source regions:

1. Every primary Key Facts item represented in the “Key facts” section.
   Treat explanatory or comparative wording embedded within a Key Facts
   item as context for that item rather than as an additional standalone
   record.

2. The principal policy-context statements represented in Summary
   paragraphs 1, 3 and 4 concerning:
   - the definition of STEM;
   - the main STEM skills-development routes;
   - departmental responsibilities for STEM skills.

3. The principal policy findings represented in Summary paragraphs
   7 to 12 and the conclusion on value for money in paragraph 21.

4. Every explicitly stated quantitative observation in Summary
   paragraphs 13 to 17 that belongs to the defined education-pipeline
   extraction scope.

5. The explicitly represented government-initiative observations in
   Summary paragraphs 18 to 20 concerning:

   - T levels and their career routes;
   - national colleges focusing on STEM skills;
   - the qualification level targeted by institutes of technology;
   - the maths and physics teacher supply package;
   - the target for recruiting additional maths and physics teachers;
   - the target for improving the skills of non-specialist teachers;
   - returning teachers recruited by the return to teaching pilot;
   - the recruitment target for that pilot;
   - returning teachers who completed the training provided.

Do not create an additional observation from comparative wording that
only describes the relationship between two already represented
quantities, such as a recruited number relative to its stated target.

6. Each recommendation represented in recommendations 22(a) to 24(f).

Exclude:

- publication metadata;
- contents-page entries;
- copyright and publisher information;
- contact details;
- website and social-media information;
- document prices;
- paragraph numbers and page numbers as observations;
- values appearing only in cross-references;
- qualitative explanatory details that are not one of the requested
  policy-context records, policy findings or recommendations;
- values that are not explicit source observations;
- calculated, derived or inferred values.

For every included record extract exactly these fields:

- Category
- Statement or Section
- Metric
- Topic
- Value
- Unit
- Qualifier
- Reporting Period
- Source Location

Category:

Use exactly one of:

- Key fact
- Policy context
- Policy finding
- Education pipeline statistic
- Government initiative
- Recommendation

Statement or Section:

- Preserve a concise source-grounded statement or section label that
  identifies where the observation belongs.
- Do not introduce external interpretation.

Metric:

- Provide a concise source-grounded name for the quantitative metric
  or policy statement represented by the record.
- For qualitative policy records, use a concise label that identifies
  the policy concept or recommendation.

Topic:

- Preserve the relevant source-grounded STEM topic, population,
  programme, institution or policy area.
- Do not merge separate observations merely because they concern a
  similar topic.

Value:

- Use a JSON number for explicitly represented quantitative values.
- Use null for qualitative policy records that do not contain a
  primary quantitative value.
- Preserve explicitly negative values as negative numbers.
- When the source explicitly describes a quantitative change as a
  fall, decrease, decline or reduction, encode the Value as a negative
  number even when the printed percentage does not contain a minus sign.
- When the source explicitly describes a quantitative change as a rise,
  increase or growth, preserve the Value as positive.
- Do not calculate, derive, convert or infer values.
- Do not rescale, calculate or convert proportions or percentages.
  The directional-sign rule above is the only permitted sign encoding.

Unit:

- Preserve the source-grounded measurement unit.
- Use null when no explicit quantitative unit applies.
- Do not place approximation or inequality wording in Unit.

Qualifier:

- Preserve explicit approximation, inequality or threshold wording
  directly associated with a quantitative value, such as:
  “around”, “almost”, “over”, “more than”, “just over”, “minimum”
  or equivalent wording represented in the source.
- Use null when no explicit qualifier is associated with the value.

Reporting Period:

- Preserve explicitly associated years, academic years, durations,
  comparison periods or relative periods.
- Use null when no explicit reporting period applies.

Source Location:

Use concise physical-PDF source locations grounded in the original
document, for example:

- PDF page 6 — Key facts
- PDF page 7 — Summary paragraph 1
- PDF page 7 — Summary paragraph 3
- PDF page 7 — Summary paragraph 4
- PDF page 8 — Summary paragraph 7
- PDF page 9 — Summary paragraph 8
- PDF page 9 — Summary paragraph 9
- PDF page 9 — Summary paragraph 10
- PDF page 9 — Summary paragraph 11
- PDF page 9 — Summary paragraph 12
- PDF page 10 — Summary paragraph 13
- PDF page 10 — Summary paragraph 14
- PDF page 10 — Summary paragraph 15
- PDF page 10 — Summary paragraph 16
- PDF page 11 — Summary paragraph 17
- PDF page 11 — Summary paragraph 18
- PDF page 11 — Summary paragraph 19
- PDF page 11 — Summary paragraph 20
- PDF page 11 — Summary paragraph 21
- PDF page 12 — Recommendation 22(a)
- PDF page 12 — Recommendation 22(b)
- PDF page 12 — Recommendation 23(c)
- PDF page 12 — Recommendation 23(d)
- PDF page 12 — Recommendation 24(e)
- PDF page 12 — Recommendation 24(f)

Additional extraction rules:

- Use both the visible layout and textual content of the original PDF.
- Respect the document's visual reading order.
- Preserve repeated observations when the same or similar statistic
  is explicitly represented in different source sections.
- Do not deduplicate distinct source observations.
- Do not use external knowledge.
- Do not follow hyperlinks.
- Do not silently correct values, units or wording.
- Do not infer missing observations.
- Verify that all content within the defined source scope has been
  processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D7",
  "branch": "A",
  "records": [
    {
      "Category": null,
      "Statement or Section": null,
      "Metric": null,
      "Topic": null,
      "Value": null,
      "Unit": null,
      "Qualifier": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
"""


PROMPT_PATH.write_text(
    BRANCH_A_PROMPT,
    encoding="utf-8"
)


PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)


print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)

print(
    BRANCH_A_PROMPT
)

## Independent Branch A extraction

Open new independent conversation.

Upload:

1. the complete original D7 PDF;
2. `D7_branch_A_prompt.txt`.

Submit the prompt once.

Save the complete, untouched model response as:

`D7_branch_A_raw_response.txt`

Upload the untouched TXT response in the next cell.

In [ ]:
# ============================================================
# 5. Raw response preservation and parsing
# ============================================================

print(
    "Upload the untouched "
    "D7_branch_A_raw_response.txt file."
)


uploaded = files.upload()


txt_paths = [
    Path(name)

    for name
    in uploaded

    if name.lower().endswith(
        ".txt"
    )
]


if len(txt_paths) != 1:

    raise ValueError(
        "Upload exactly one TXT raw-response file."
    )


UPLOADED_RAW_RESPONSE_PATH = (
    txt_paths[0]
)


raw_response_text = (
    UPLOADED_RAW_RESPONSE_PATH.read_text(
        encoding="utf-8"
    )
)


if not raw_response_text.strip():

    raise ValueError(
        "The uploaded raw response is empty."
    )

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = (
    sha256_file(
        RAW_RESPONSE_PATH
    )
)

valid_json = False

json_parsing_error = None

parsed_response = None


try:

    parsed_response = json.loads(
        raw_response_text
    )

    valid_json = True


except json.JSONDecodeError as error:

    json_parsing_error = str(
        error
    )

top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)


document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)


document_id_correct = (
    top_level_object_valid
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)


branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)


branch_correct = (
    top_level_object_valid
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)


records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)


records_is_list = (
    top_level_object_valid
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)

records_evaluable = (
    valid_json
    and top_level_object_valid
    and records_present
    and records_is_list
)


if records_evaluable:

    extracted_records = (
        parsed_response[
            "records"
        ]
    )

    observed_record_count = len(
        extracted_records
    )


else:

    extracted_records = []

    observed_record_count = None

parsed_extraction_created = False

parsed_extraction_sha256 = None


if records_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get(
                "document_id"
            ),

        "branch":
            parsed_response.get(
                "branch"
            ),

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_created = True

    parsed_extraction_sha256 = (
        sha256_file(
            PARSED_EXTRACTION_PATH
        )
    )


print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)

print(
    "Valid JSON:",
    valid_json
)

print(
    "JSON parsing error:",
    json_parsing_error
)

print(
    "Top-level object valid:",
    top_level_object_valid
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed record count:",
    observed_record_count
)


if records_evaluable:

    extracted_df = pd.DataFrame(
        extracted_records
    )

    display(
        extracted_df.head(
            10
        )
    )

In [ ]:
# ============================================================
# 6. Record and content diagnostics
# ============================================================

record_structure_issues = []

field_type_issues = []

missing_mandatory_values = []


# ------------------------------------------------------------
# Record schema
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Record is not a JSON object"
                }
            )

            continue


        observed_fields = list(
            record.keys()
        )


        if (
            observed_fields
            != EXPECTED_FIELDS
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        (
                            "Field names or field "
                            "order differ"
                        ),

                    "expected_fields":
                        EXPECTED_FIELDS,

                    "observed_fields":
                        observed_fields,

                    "missing_fields":
                        [
                            field

                            for field
                            in EXPECTED_FIELDS

                            if field
                            not in record
                        ],

                    "extra_fields":
                        [
                            field

                            for field
                            in observed_fields

                            if field
                            not in EXPECTED_FIELDS
                        ]
                }
            )


    records_with_structure_issues = len({
        issue[
            "record_index"
        ]

        for issue
        in record_structure_issues
    })


    record_schema_valid = (
        records_with_structure_issues
        == 0
    )


else:

    records_with_structure_issues = None

    record_schema_valid = None


# ------------------------------------------------------------
# Field types
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            continue


        for field in STRING_OR_NULL_FIELDS:

            value = record.get(
                field
            )


            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):

                field_type_issues.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(
                                value
                            ).__name__,

                        "expected_type":
                            "string or null"
                    }
                )


        for field in NUMERIC_OR_NULL_FIELDS:

            value = record.get(
                field
            )


            if (
                isinstance(
                    value,
                    bool
                )
                or (
                    value is not None
                    and not isinstance(
                        value,
                        (
                            int,
                            float
                        )
                    )
                )
            ):

                field_type_issues.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(
                                value
                            ).__name__,

                        "expected_type":
                            "number or null"
                    }
                )


        # ----------------------------------------------------
        # Mandatory-content diagnostic
        #
        # Kept separate from schema/type validity.
        # ----------------------------------------------------

        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(
                field
            )


            if (
                value is None
                or value == ""
            ):

                missing_mandatory_values.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field
                    }
                )


    records_with_type_issues = len({
        issue[
            "record_index"
        ]

        for issue
        in field_type_issues
    })


    field_types_valid = (
        records_with_type_issues
        == 0
    )


    missing_mandatory_value_count = len(
        missing_mandatory_values
    )


    mandatory_fields_complete = (
        missing_mandatory_value_count
        == 0
    )


else:

    records_with_type_issues = None

    field_types_valid = None

    missing_mandatory_value_count = None

    mandatory_fields_complete = None


# ------------------------------------------------------------
# Record-count and category diagnostics
# ------------------------------------------------------------

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )

            for record
            in extracted_records

            if isinstance(
                record,
                dict
            )
        )
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


else:

    record_count_valid = None

    observed_category_counts = None

    categories_valid = None

    category_counts_valid = None


# ------------------------------------------------------------
# Exact-complete-record duplicate diagnostic
# ------------------------------------------------------------

if records_evaluable:

    duplicate_counter = Counter(
        tuple(
            record.get(
                field
            )

            for field
            in EXPECTED_FIELDS
        )

        for record
        in extracted_records

        if isinstance(
            record,
            dict
        )
    )


    duplicate_records = [
        list(
            key
        )

        for key, count
        in duplicate_counter.items()

        if count > 1
    ]


    duplicate_record_count = len(
        duplicate_records
    )


    duplicate_records_absent = (
        duplicate_record_count
        == 0
    )


else:

    duplicate_records = None

    duplicate_record_count = None

    duplicate_records_absent = None


# ------------------------------------------------------------
# D7-specific content diagnostics
# ------------------------------------------------------------

if records_evaluable:

    negative_value_count = sum(
        1

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and isinstance(
                record.get(
                    "Value"
                ),
                (
                    int,
                    float
                )
            )

            and not isinstance(
                record.get(
                    "Value"
                ),
                bool
            )

            and record.get(
                "Value"
            ) < 0
        )
    )


    null_value_count = sum(
        1

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and record.get(
                "Value"
            )
            is None
        )
    )


    qualifier_count = sum(
        1

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and record.get(
                "Qualifier"
            )
            is not None
        )
    )


else:

    negative_value_count = None

    null_value_count = None

    qualifier_count = None


CONTENT_DIAGNOSTICS = {
    "record_count_matches_reference":
        record_count_valid,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records_absent":
        duplicate_records_absent,

    "negative_value_count":
        negative_value_count,

    "null_value_count":
        null_value_count,

    "qualifier_count":
        qualifier_count
}


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Records with structure issues:",
    records_with_structure_issues
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Records with type issues:",
    records_with_type_issues
)

print(
    "Observed records:",
    observed_record_count
)

print(
    "Record count matches reference:",
    record_count_valid
)

print(
    "Category counts match:",
    category_counts_valid
)

print(
    "Missing mandatory values:",
    missing_mandatory_value_count
)

print(
    "Duplicate complete records:",
    duplicate_record_count
)

print(
    "Qualifiers observed:",
    qualifier_count
)

print(
    "\nObserved category counts:"
)

print(
    json.dumps(
        observed_category_counts,
        ensure_ascii=False,
        indent=2
    )
    if observed_category_counts is not None
    else None
)

In [ ]:
# ============================================================
# 7. Technical diagnostic summary and experiment metadata
# ============================================================

STRUCTURAL_CHECKS = {
    "valid_json":
        bool(
            valid_json
        ),

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_present":
        bool(
            document_id_present
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_present":
        bool(
            branch_present
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_present":
        bool(
            records_present
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "record_schema_valid":
        (
            record_schema_valid
            if records_evaluable
            else None
        ),

    "field_types_valid":
        (
            field_types_valid
            if records_evaluable
            else None
        )
}


structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])


TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "valid_json":
        valid_json,

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        top_level_object_valid,

    "document_id_present":
        document_id_present,

    "document_id_correct":
        document_id_correct,

    "branch_present":
        branch_present,

    "branch_correct":
        branch_correct,

    "records_present":
        records_present,

    "records_is_list":
        records_is_list,

    "records_evaluable":
        records_evaluable,

    "structural_checks":
        STRUCTURAL_CHECKS,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_valid":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_valid":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issue_count":
        (
            len(
                field_type_issues
            )
            if records_evaluable
            else None
        ),

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "missing_mandatory_values":
        (
            missing_mandatory_values
            if records_evaluable
            else None
        ),

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records":
        duplicate_records,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        )
}


TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment metadata
# ------------------------------------------------------------

EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "source_structure": {
        "expected_page_count":
            EXPECTED_PAGE_COUNT,

        "observed_page_count":
            PAGE_COUNT,

        "page_count_verified":
            PAGE_COUNT_VALID,

        "machine_readable_text_layer":
            TEXT_EXTRACTABLE,

        "expected_components_verified":
            DOCUMENT_GROUNDING_VALID
    },

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "diagnostic_text_extraction_applied":
        True,

    "text_extraction_used_as_model_input":
        False,

    "pdf_to_text_conversion_applied":
        False,

    "ocr_applied":
        False,

    "page_cropping_applied":
        False,

    "page_extraction_applied":
        False,

    "layout_reconstruction_applied":
        False,

    "table_conversion_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_category_counts":
            EXPECTED_CATEGORY_COUNTS,

        "expected_fields":
            EXPECTED_FIELDS
    },

    "reference_expectations_disclosed_to_model":
        False,

    "input_integrity_file":
        INPUT_INTEGRITY_PATH.name,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED,

    "representation_file":
        REPRESENTATION_PATH.name,

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        (
            "JSON object with document_id, "
            "branch and records"
        ),

    "execution_environment":
        "Independent ChatGPT conversation",

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        ),

    "notes": (
        "Branch A submits the complete original D7 PDF "
        "directly to the model. PyMuPDF text extraction "
        "is used only for source-integrity diagnostics. "
        "No PDF-to-text conversion, OCR, structural "
        "conversion, layout reconstruction, cleaning, "
        "normalisation or manual correction is applied "
        "before extraction. Stage 1 reference values and "
        "expected record counts are not supplied to the "
        "model. Content-level validation is performed "
        "separately in Validation A — D7."
    )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment summary
# ------------------------------------------------------------

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        INPUT_INTEGRITY_PASSED,

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "negative_value_count":
        negative_value_count,

    "null_value_count":
        null_value_count,

    "qualifier_count":
        qualifier_count,

    "raw_response_preserved":
        RAW_RESPONSE_PATH.exists(),

    "parsed_extraction_created":
        parsed_extraction_created,

    "content_validation_performed":
        False,

    "notes": (
        "This notebook performs source verification, "
        "D7 Branch A direct-ingestion response preservation "
        "and technical diagnostics only. Agreement with the "
        "fixed Stage 1 reference dataset is evaluated "
        "separately in Validation A — D7."
    )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "Structural checks:"
)

print(
    json.dumps(
        STRUCTURAL_CHECKS,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "\nContent diagnostics:"
)

print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "\nStructurally evaluable:",
    structurally_evaluable
)

print(
    "\nExperiment summary:"
)

print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 8. Final experiment status and artefact inventory
# ============================================================

print(
    "=" * 60
)

print(
    "D7 Branch A experiment completed"
)

print(
    "=" * 60
)


print(
    "Input integrity passed       :",
    INPUT_INTEGRITY_PASSED
)

print(
    "Raw response preserved       :",
    RAW_RESPONSE_PATH.exists()
)

print(
    "Valid JSON                   :",
    valid_json
)

print(
    "Records evaluable            :",
    records_evaluable
)

print(
    "Expected records             :",
    EXPECTED_RECORD_COUNT
)

print(
    "Observed records             :",
    (
        observed_record_count
        if observed_record_count
        is not None
        else "Not evaluable"
    )
)

print(
    "Record count matches         :",
    record_count_valid
)

print(
    "Category counts match        :",
    category_counts_valid
)

print(
    "Record schema valid          :",
    record_schema_valid
)

print(
    "Field types valid            :",
    field_types_valid
)

print(
    "Structurally evaluable       :",
    structurally_evaluable
)

print(
    "Content validation performed : False"
)

print(
    "Next step                    : Validation A — D7"
)


required_output_paths = [
    INPUT_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]


if (
    parsed_extraction_created
    and PARSED_EXTRACTION_PATH.exists()
):

    required_output_paths.append(
        PARSED_EXTRACTION_PATH
    )


missing_output_files = [
    path.name

    for path
    in required_output_paths

    if not path.exists()
]


if missing_output_files:

    raise AssertionError(
        "Missing output files: "
        f"{missing_output_files}"
    )


print(
    "\nGenerated D7 Branch A files:\n"
)


for path in required_output_paths:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )